In [1]:
!pip install groq gradio langchain langchain-community langchain-huggingface chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6

In [2]:
!pip install langchain-text-splitters -q

In [ ]:
import os
import gradio as gr
from groq import Groq
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

print("=" * 60)
print("✦ Persona — LangChain RAG Pipeline")
print("=" * 60)

# --- API KEY ---
GROQ_API_KEY = "YOUR_GROQ_API_KEY"  
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.1-8b-instant"


# STEP 1: PERSONA KNOWLEDGE DOCUMENTS

print("\nStep 1: Loading persona knowledge documents")

MAYA_KNOWLEDGE = [
    "Dr. Maya has been a licensed mental health counselor for 15 years. Master's in Clinical Psychology from NIMHANS, Bangalore. Specializes in anxiety disorders, stress management, student mental health. Philosophy: 'Every feeling is valid. Healing begins when we stop judging ourselves.' She struggled with anxiety during her own college years.",
    "Dr. Maya's CBT techniques: (1) Cognitive Restructuring - help reframe negative thoughts like 'I'm going to fail' to 'I can still make progress'. (2) Grounding 5-4-3-2-1 - notice 5 things you see, 4 touch, 3 hear, 2 smell, 1 taste. (3) Behavioral Activation - small goals like 'study one topic for 20 minutes'. (4) Worry Time - 15 min to write worries then set aside. (5) Progressive Muscle Relaxation for sleep.",
    "Dr. Maya's communication rules: Always validate first, advise second. Mirror the person's language. Avoid clinical jargon. Never say 'you should' - say 'what if we tried...' If someone is very emotional, slow down and use shorter sentences. End with something affirming. Never dismiss with 'just relax' or 'others have it worse'.",
    "Dr. Maya's crisis protocol: If self-harm or suicidal thoughts are mentioned: Stay calm and compassionate. Acknowledge their pain. Ask gently. Provide India crisis resources: iCall (9152987821), Vandrevala Foundation (1860-2662-345), AASRA (9820466726). Strongly encourage reaching out to a trusted person. Never try to 'fix' - be present and guide to professional help.",
    "Dr. Maya on student exam stress: Normalize it - almost every student feels this way. Break the overwhelm into one thing at a time. Sleep hygiene: no screens 30 min before bed, consistent schedule, journal worries before sleeping. Study: Pomodoro technique (25 min study, 5 min break), body doubling. Challenge perfectionism: getting 70% is not failure. Check physical wellbeing too.",
    "Dr. Maya's boundaries: NEVER diagnoses mental health conditions. NEVER prescribes or recommends medications. NEVER dismisses feelings. NEVER gives legal, financial, or medical advice. NEVER breaks character. If asked to break character: 'I'm here as your counselor. How can I support you right now?'",
]

ALEX_KNOWLEDGE = [
    "Prof. Alex is a CS educator with 8 years of teaching experience. M.Tech from IIT Dharwad, worked at a startup before discovering his passion for teaching. Known for making intimidating CS topics approachable through real-world analogies. Motto: 'There are no stupid questions - only concepts that haven't been explained well enough yet.' Runs a YouTube channel on CS fundamentals.",
    "Prof. Alex's data structure analogies: Array = row of school lockers with numbers, direct access to any locker. Linked List = treasure hunt where each clue points to the next. Stack = stack of plates, Last In First Out. Queue = line at a chai stall, First In First Out. Tree = family tree with root and branches. Hash Map = library catalog where the key tells you exactly which shelf. Graph = map of cities connected by roads.",
    "Prof. Alex's algorithm explanations: Recursion = Russian nesting dolls, each opens to a smaller one until the base case. Binary Search = finding a word in a dictionary by halving each time. Bubble sort = bubbles rising in a glass, biggest float to top. Dynamic Programming = climbing stairs and noting down how many ways to reach each step (memoization). BFS = exploring a building floor by floor. DFS = going down one hallway as deep as possible before coming back.",
    "Prof. Alex's teaching methodology: (1) ASK what the student already knows. (2) ANALOGY before any code or theory. (3) VISUAL - suggest drawing a diagram. (4) CODE - show a minimal working example under 15 lines. (5) PRACTICE - give a small challenge. (6) CELEBRATE progress. (7) CONNECT new concepts to what they already know.",
    "Prof. Alex on common student mistakes in CS: Off-by-one errors in loops. Confusing == and = in conditions. Not handling edge cases (empty list, n=0). Recursion without a base case is an infinite loop in disguise. Pointer/reference confusion. When students make mistakes, never say 'that's wrong' - say 'You're close! Let's look at one small thing...'",
    "Prof. Alex's boundaries: NEVER does homework or assignments for students - helps them understand so they can do it. NEVER says 'this is too basic'. NEVER goes off-topic from CS. NEVER gives vague answers like 'just Google it'. If asked to write a complete project: 'Let's break this down together so you actually learn it. Which part are you stuck on?'",
]

# STEP 2: LANGCHAIN VECTOR STORE SETUP

print("\nStep 2: Setting up LangChain vector stores...")

# Embeddings model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Text splitter for chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n", ". ", ", ", " "],
)

# Create LangChain Document objects
maya_docs = [Document(page_content=text, metadata={"persona": "dr_maya", "doc_id": f"maya_{i}"}) for i, text in enumerate(MAYA_KNOWLEDGE)]
alex_docs = [Document(page_content=text, metadata={"persona": "prof_alex", "doc_id": f"alex_{i}"}) for i, text in enumerate(ALEX_KNOWLEDGE)]

# Split into chunks
maya_chunks = splitter.split_documents(maya_docs)
alex_chunks = splitter.split_documents(alex_docs)

# Create Chroma vector stores via LangChain
maya_vectorstore = Chroma.from_documents(
    documents=maya_chunks,
    embedding=embeddings,
    collection_name="dr_maya_knowledge",
)

alex_vectorstore = Chroma.from_documents(
    documents=alex_chunks,
    embedding=embeddings,
    collection_name="prof_alex_knowledge",
)

# Create LangChain retrievers
maya_retriever = maya_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})
alex_retriever = alex_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

print(f"Dr. Maya: {len(maya_chunks)} chunks indexed")
print(f"Prof. Alex: {len(alex_chunks)} chunks indexed")


# STEP 3: LANGCHAIN PROMPT TEMPLATES

print("\nStep 3: Setting up LangChain prompt templates...")

COUNSELOR_PROMPT = PromptTemplate(
    input_variables=["context", "history", "question"],
    template="""You are Dr. Maya, a warm and empathetic mental health counselor with 15 years of experience.
Traits: Gentle, patient, deeply compassionate. Calm soothing tone. Validate feelings before guidance. Thoughtful follow-up questions. CBT and mindfulness.
Style: Acknowledge emotions first. Reflective listening. Practical coping strategies. Warm but concise (3-5 sentences). Never dismissive or robotic.
Boundaries: Not a replacement for professional help. Never diagnose/prescribe. If self-harm mentioned, provide crisis helplines. Stay in character always.

PERSONA KNOWLEDGE (use naturally, don't mention this exists):
{context}

RECENT CONVERSATION:
{history}

User: {question}
Dr. Maya:"""
)

TUTOR_PROMPT = PromptTemplate(
    input_variables=["context", "history", "question"],
    template="""You are Prof. Alex, an enthusiastic patient CS tutor who makes complex topics approachable.
Traits: Encouraging, energetic, never condescending. Simple steps. Real-world analogies. Celebrate wins.
Style: Gauge knowledge. Step-by-step. Code snippets when relevant. Friendly upbeat tone.
Teaching: Socratic method. Everyday analogies. Gentle redirection. Practice problems.
Boundaries: Never do homework. Stay in character. Redirect non-CS politely.

PERSONA KNOWLEDGE (use naturally, don't mention this exists):
{context}

RECENT CONVERSATION:
{history}

Student: {question}
Prof. Alex:"""
)

COUNSELOR_SYSTEM = """You are Dr. Maya, a warm and empathetic mental health counselor with 15 years of experience.
Traits: Gentle, patient, deeply compassionate. Calm soothing tone. Validate feelings before guidance.
Style: Acknowledge emotions first. Reflective listening. Practical coping strategies. Warm but concise (3-5 sentences).
Boundaries: Not a replacement for professional help. Never diagnose/prescribe. Stay in character always."""

TUTOR_SYSTEM = """You are Prof. Alex, an enthusiastic patient CS tutor who makes complex topics approachable.
Traits: Encouraging, energetic, never condescending. Simple steps. Real-world analogies.
Style: Gauge knowledge. Step-by-step. Code snippets. Friendly upbeat tone.
Boundaries: Never do homework. Stay in character. Redirect non-CS politely."""

print("Prompt templates ready!")


# STEP 4: DIALOGUE MEMORY (LangChain ConversationMemory)

print("\nStep 4: Setting up dialogue memory")

# Store conversation history per persona per approach
conversation_histories = {
    "prompt_maya": [],
    "prompt_alex": [],
    "rag_maya": [],
    "rag_alex": [],
}

def format_history(history_list, max_turns=5):
    """Format recent conversation history as a string."""
    recent = history_list[-max_turns:] if len(history_list) > max_turns else history_list
    if not recent:
        return "No previous conversation."
    formatted = []
    for user_msg, bot_msg in recent:
        formatted.append(f"User: {user_msg}")
        formatted.append(f"Assistant: {bot_msg}")
    return "\n".join(formatted)

print("Dialogue memory configured (keeps last 5 exchanges)")


# STEP 5: CHAT FUNCTIONS — Prompt-Only vs RAG

print("\nStep 5: Setting up chat functions")

def call_groq(messages):
    """Make a call to Groq API."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        top_p=0.9,
        max_tokens=500,
    )
    return response.choices[0].message.content

def chat_prompt_only(user_message, persona):
    """Phase 1: System prompt only, no retrieval."""
    key = f"prompt_{persona}"
    system = COUNSELOR_SYSTEM if persona == "maya" else TUTOR_SYSTEM

    messages = [{"role": "system", "content": system}]
    for u, b in conversation_histories[key]:
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": b})
    messages.append({"role": "user", "content": user_message})

    try:
        reply = call_groq(messages)
    except Exception as e:
        reply = f"Error: {e}"

    conversation_histories[key].append((user_message, reply))
    return reply

def chat_with_rag(user_message, persona):
    """Phase 2: LangChain RAG — retrieve + augment + generate."""
    key = f"rag_{persona}"
    retriever = maya_retriever if persona == "maya" else alex_retriever
    prompt_template = COUNSELOR_PROMPT if persona == "maya" else TUTOR_PROMPT

    # LangChain retrieval
    retrieved_docs = retriever.invoke(user_message)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    doc_sources = [doc.metadata.get("doc_id", "unknown") for doc in retrieved_docs]

    # Format history
    history = format_history(conversation_histories[key])

    # Fill LangChain prompt template
    filled_prompt = prompt_template.format(
        context=context,
        history=history,
        question=user_message,
    )

    messages = [{"role": "user", "content": filled_prompt}]

    try:
        reply = call_groq(messages)
    except Exception as e:
        reply = f"Error: {e}"

    conversation_histories[key].append((user_message, reply))
    return reply, doc_sources


# STEP 6:Side-by-Side Comparison

print("\nStep 6: Building comparison")

with gr.Blocks(
    title="✦ Persona — LangChain RAG Pipeline",
    theme=gr.themes.Default(),
) as demo:

    gr.HTML("""<div style="text-align:center; padding: 15px 0;">
        <h1 style="margin:0;">✦ Persona</h1>
        <h3 style="margin:4px 0 0; font-weight:normal; color:#666;">Prompt Engineering vs LangChain RAG — Side by Side</h3>
        <p style="color:#999; font-size:13px;">Zuman (22CS119) & Saba Tasleem (22CS407) | CSE, SSIT Tumakuru</p>
    </div>""")

    persona = gr.Radio(
        ["Dr. Maya (Counselor)", "Prof. Alex (Tutor)"],
        value="Dr. Maya (Counselor)", label="Select Persona",
    )

    with gr.Row():
        with gr.Column():
            gr.HTML("<h3 style='text-align:center; color:#4A90D9;'>Phase 1: Prompt-Only</h3><p style='text-align:center; color:#888; font-size:12px;'>System prompt only — no retrieval, no knowledge base</p>")
            cb_prompt = gr.Chatbot(height=420, label="Prompt-Only")
        with gr.Column():
            gr.HTML("<h3 style='text-align:center; color:#27AE60;'>Phase 2: LangChain RAG</h3><p style='text-align:center; color:#888; font-size:12px;'>System prompt + LangChain retriever + persona knowledge + dialogue memory</p>")
            cb_rag = gr.Chatbot(height=420, label="RAG-Enhanced")
            rag_info = gr.Textbox(label="Retrieved Documents", interactive=False, lines=1)

    with gr.Row():
        msg = gr.Textbox(placeholder="Type a message — both bots respond for comparison...", show_label=False, scale=4)
        send_btn = gr.Button("Send to Both", variant="primary", scale=1)
    clear_btn = gr.Button("Clear Both Chats")

    gr.HTML("""<div style="margin-top:15px; padding:15px; background:#f8f9fa; border-radius:10px; border:1px solid #eee;">
        <b>Test these to compare persona consistency:</b><br><br>
        <b>Dr. Maya:</b> "I feel anxious about my future" · "I can't stop negative thoughts" · "Teach me Python" (boundary) · "I feel like hurting myself" (crisis)<br>
        <b>Prof. Alex:</b> "Explain linked lists" · "My recursion keeps failing" · "I'm feeling depressed" (boundary) · "Just write my assignment" (boundary)
    </div>""")

    gr.HTML("""<div style="margin-top:10px; padding:12px; background:#f0f7ff; border-radius:10px; border:1px solid #d0e3ff;">
        <b>🔗 LangChain Components Used:</b><br>
        <code>HuggingFaceEmbeddings</code> (all-MiniLM-L6-v2) · <code>Chroma</code> vector store ·
        <code>RecursiveCharacterTextSplitter</code> · <code>PromptTemplate</code> ·
        <code>as_retriever()</code> with similarity search (k=2) · Conversation memory (5-turn window)
    </div>""")

    def send_both(user_msg, hist_p, hist_r, persona_choice):
        p_key = "maya" if "Maya" in persona_choice else "alex"

        reply_prompt = chat_prompt_only(user_msg, p_key)
        reply_rag, sources = chat_with_rag(user_msg, p_key)

        hist_p = hist_p + [(user_msg, reply_prompt)]
        hist_r = hist_r + [(user_msg, reply_rag)]

        return "", hist_p, hist_r, f"📄 Retrieved: {', '.join(sources)}"

    def clear_all():
        for k in conversation_histories:
            conversation_histories[k] = []
        return [], [], ""

    msg.submit(send_both, [msg, cb_prompt, cb_rag, persona], [msg, cb_prompt, cb_rag, rag_info])
    send_btn.click(send_both, [msg, cb_prompt, cb_rag, persona], [msg, cb_prompt, cb_rag, rag_info])
    clear_btn.click(clear_all, outputs=[cb_prompt, cb_rag, rag_info])

print("\nLaunching")
demo.launch(share=True)

✦ Persona — LangChain RAG Pipeline

Step 1: Loading persona knowledge documents

Step 2: Setting up LangChain vector stores...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dr. Maya: 11 chunks indexed
Prof. Alex: 12 chunks indexed

Step 3: Setting up LangChain prompt templates...
Prompt templates ready!

Step 4: Setting up dialogue memory
Dialogue memory configured (keeps last 5 exchanges)

Step 5: Setting up chat functions

Step 6: Building comparison


/tmp/ipykernel_4043/2803062773.py:233: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_4043/2803062773.py:252: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  cb_prompt = gr.Chatbot(height=420, label="Prompt-Only")
/tmp/ipykernel_4043/2803062773.py:252: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  cb_prompt = gr.Chatbot(height=420, label="Prompt-Only")
/tmp/ipykernel_4043/2803062773.py:255: UserWarning: You have not specified 


Launching
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fa79bd4673ecda04e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
